# Try_014: GPTQ W4A16 + MANTA+KMMLU 캘리브레이션 (로컬 수정본)

**기준: Try_000 (Phase2=0.5914 베스트)**  
**전략: PerfNorm 극대화 — 한국어 벤치마크 분포 정렬 캘리브레이션**

### Try_000 vs Try_014 (vs Try_014_colab 문제점 수정)
| 항목 | Try_000 | Try_014_colab | Try_014 (이 파일) |
|------|---------|---------------|-------------------|
| 캘리브레이션 | MANTA 128 | MANTA 160 (KMMLU 실패) | MANTA 128 + **KMMLU 32** = 160 |
| lm_head 저장 | 단일 (930MB) | 중복 1.4GB ❌ | **tie_weights() → 단일** ✅ |
| zip 경로 | `model/` ✅ | `drive/MyDrive/model/` ❌ | **`model/` 보장** ✅ |
| 실행 환경 | 로컬 | Colab | **로컬 (MPS/CUDA/CPU)** |

### 핵심 수정 사항 (vs Try_014_colab)
1. **KMMLU 로드 수정**: `"all"` config 없음 → 과목별 subset 방식으로 변경
2. **tie_weights() 호출**: lm_head=embed_tokens 연결 복원 → 파일 크기 ~930MB
3. **zip 경로 보장**: `root_dir=parent`, `base_dir="model"` 명시
4. **생성 검증 셀 추가**: unique_ratio < 0.3이면 반복 루프 경보
5. **로컬 환경**: Colab 의존 코드 완전 제거

예상 소요 시간: ~1.5시간 (CPU) / ~20분 (CUDA GPU)

In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f"전체 RAM: {ram.total / 1e9:.1f} GB")
print(f"사용 가능: {ram.available / 1e9:.1f} GB")

In [ ]:
import os
import gc
import itertools
import torch
import shutil
from pathlib import Path

from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print("[INFO] 임포트 완료")

In [ ]:
# =============================================
# GPU 디바이스 자동 감지
# ⚠️ llmcompressor oneshot()은 CUDA/XPU만 지원
#    → MPS Mac에서는 GPTQ가 CPU로 강제 실행됨
# =============================================
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("[INFO] ✅ Apple MPS (GPU) 감지")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"[INFO] ✅ CUDA GPU 감지: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu")
    print("[INFO] CPU 사용 (GPU 없음)")

print(f"[INFO] DEVICE = {DEVICE}")
print("[INFO] ⚠️  GPTQ(oneshot)는 llmcompressor 제약으로 CPU/CUDA 실행")

In [ ]:
# =============================================
# 하이퍼파라미터
# =============================================
MODEL_ID = "./base_model"   # 로컬 경로 (Colab 경로 아님)
OUT_DIR  = "./model"

DATASET_ID    = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ─── 캘리브레이션 샘플 구성 (총 160개) ──────
# ★ MMLU/GSM8K 제외 — 성능 저하 확인됨
# ★ MANTA(메인) + KMMLU(보조) 2-소스 전략
N_MANTA  = 128   # 한국어 대화 (메인, 베이스)
N_KMMLU  = 32    # 한국어 MCQ  (보조, KMMLU-PRO/REDUX 평가 반영)
NUM_CALIBRATION_SAMPLES = N_MANTA + N_KMMLU  # 160

MAX_SEQUENCE_LENGTH = 512

# ─── GPTQ 설정 ──────────────────────────────
SCHEME  = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]  # tie_word_embeddings=True → 동일 텐서

# ─── KMMLU 과목 (Colab에서 "all" config 없음 → 과목별 로드) ────
KMMLU_SUBJECTS = [
    "Korean-History", "Law", "Computer-Science",
    "Management", "Health", "Education"
]

print("[INFO] 설정 완료")
print(f"  모델: {MODEL_ID}")
print(f"  캘리브레이션: MANTA {N_MANTA} + KMMLU {N_KMMLU} = {NUM_CALIBRATION_SAMPLES}개")
print(f"  양자화: GPTQ W4A16 (group_size=128)")

In [ ]:
print("[INFO] 모델 로드 중 (bfloat16)...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,       # torch_dtype deprecated → dtype 사용
    trust_remote_code=True,
)

print(f"[INFO] 로드 완료 | 레이어: {model.config.num_hidden_layers}")
print(f"  tie_word_embeddings: {model.config.tie_word_embeddings}")

In [ ]:
# =============================================
# 캘리브레이션 데이터 로드
# MANTA 128 (한국어 대화) + KMMLU 32 (한국어 MCQ)
# ★ KMMLU: "all" config 없음 → 과목별 subset 로드로 수정
# =============================================
print("[INFO] 캘리브레이션 데이터 로드 중...")
all_samples = []

# ── 1. MANTA 128개 ────────────────────────────────────
print(f"  [1/2] MANTA {N_MANTA}개 로드 중...")
manta_ds = load_dataset(DATASET_ID, split=f"train[:{N_MANTA}]")
for ex in manta_ds:
    text = tokenizer.apply_chat_template(
        ex["conversations"], add_generation_prompt=True, tokenize=False
    )
    all_samples.append({"text": text})
print(f"      → {len(all_samples)}개")

# ── 2. KMMLU 32개 (과목별 순환 로드) ─────────────────────
# "all" config 없음 → 과목 순서대로 per_subject개씩 로드
print(f"  [2/2] KMMLU {N_KMMLU}개 로드 중... (과목별 로드)")
per_subject = max(1, N_KMMLU // len(KMMLU_SUBJECTS))  # 과목당 샘플 수
kmmlu_total = 0

for subj in KMMLU_SUBJECTS:
    if kmmlu_total >= N_KMMLU:
        break
    remain = N_KMMLU - kmmlu_total
    n_this = min(per_subject, remain)
    try:
        kds = load_dataset("HAERAE-HUB/KMMLU", subj, split=f"test[:{n_this}]")
        for item in kds:
            q = item.get("question", item.get("Question", ""))
            choices = "".join(
                f"\n{k}. {item[k]}" for k in ["A","B","C","D"] if item.get(k)
            )
            messages = [{"role": "user", "content": q + choices}]
            text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
            all_samples.append({"text": text})
            kmmlu_total += 1
        print(f"      {subj}: {n_this}개")
    except Exception as e:
        print(f"      {subj}: 실패 ({e})")

# KMMLU 부족분은 MANTA 추가 보충
if kmmlu_total < N_KMMLU:
    deficit = N_KMMLU - kmmlu_total
    print(f"      ⚠ KMMLU {deficit}개 부족 → MANTA 추가 보충")
    extra_ds = load_dataset(DATASET_ID, split=f"train[{N_MANTA}:{N_MANTA+deficit}]")
    for ex in extra_ds:
        text = tokenizer.apply_chat_template(
            ex["conversations"], add_generation_prompt=True, tokenize=False
        )
        all_samples.append({"text": text})

ds = Dataset.from_list(all_samples[:NUM_CALIBRATION_SAMPLES])
print(f"\n[INFO] 캘리브레이션 데이터 완료: {len(ds)}개")
print(f"  MANTA {N_MANTA} + KMMLU {kmmlu_total}개 (부족분 MANTA 보충)")

In [9]:
print("[INFO] 모델 로드 중 (bfloat16)...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
)

print(f"[INFO] 로드 완료 | 레이어: {model.config.num_hidden_layers}")

[INFO] 모델 로드 중 (bfloat16)...


`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 로드 완료 | 레이어: 30


In [ ]:
# =============================================
# 생성 검증 — 양자화 품질 확인
# unique_ratio < 0.3 → 반복 루프 → 재실행 필요
# =============================================
print("[INFO] 생성 검증 시작...")

test_prompts = [
    [{"role": "user", "content": "한국의 수도는 어디인가요?"}],
    [{"role": "user", "content": "다음 중 맞는 것을 고르세요. A. 지구는 평평하다 B. 지구는 둥글다 C. 지구는 삼각형이다 D. 지구는 네모다"}],
    [{"role": "user", "content": "Python에서 리스트를 정렬하는 함수는 무엇인가요?"}],
]

model.eval()
if torch.cuda.is_available():
    gen_device = "cuda"
elif torch.backends.mps.is_available():
    gen_device = "mps"
else:
    gen_device = "cpu"
model.to(gen_device)

all_pass = True
for i, msgs in enumerate(test_prompts):
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=40, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    unique_ratio = len(set(gen_ids.tolist())) / max(len(gen_ids), 1)

    status = "✅" if unique_ratio >= 0.3 else "❌ 반복 루프"
    if unique_ratio < 0.3:
        all_pass = False
    print(f"  [{i+1}] unique_ratio={unique_ratio:.2f} {status}")
    print(f"       → {gen_text[:80]}")

if all_pass:
    print("\n[INFO] 생성 검증 통과 — 저장 진행")
else:
    print("\n[WARN] ❌ 반복 루프 감지 — 캘리브레이션 데이터 또는 설정 재검토 필요")

In [ ]:
# =============================================
# 모델 저장
# ★ 핵심: tie_weights() 호출 → lm_head=embed_tokens 연결 복원
#   → 중복 저장 방지: 파일 크기 ~930MB (Try_014_colab의 1.4GB 문제 해결)
# =============================================
os.makedirs(OUT_DIR, exist_ok=True)

# ── tie_weights 복원 ──────────────────────────
if hasattr(model, "tie_weights"):
    model.tie_weights()
    print("[INFO] tie_weights() 완료 — lm_head/embed_tokens 연결 확인")

# ── 저장 ──────────────────────────────────────
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

gc.collect()

# ── 파일 크기 검증 ────────────────────────────
st = Path(OUT_DIR) / "model.safetensors"
size_mb = st.stat().st_size / 1e6 if st.exists() else 0
print(f"[INFO] 모델 저장 완료: {OUT_DIR}")
print(f"  safetensors 크기: {size_mb:.0f} MB")
if size_mb > 1200:
    print("  ⚠ 파일이 1.2GB 초과 — lm_head 중복 저장 가능성. tie_weights() 재확인 필요")
else:
    print("  ✅ 크기 정상 (~930MB 예상)")

In [ ]:
# =============================================
# ZIP 생성 — 제출용
# ★ 핵심 수정: root_dir과 base_dir을 명시하여 'model/' 루트 구조 보장
#   Try_014_colab 문제: root_dir="." + abs path → 'drive/MyDrive/model/' 오류
#   수정: root_dir=OUT_DIR의 부모, base_dir='model' (폴더명만)
# =============================================
zip_name = "Try_014"
out_path = Path(OUT_DIR).resolve()   # 절대 경로 확보

print(f"[INFO] {zip_name}.zip 생성 중...")
print(f"  모델 경로: {out_path}")
print(f"  zip 내부 구조: {out_path.name}/  ← 'model/' 루트 보장")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=str(out_path.parent),   # ./  (OUT_DIR의 부모)
    base_dir=str(out_path.name),     # model  (폴더명만)
)

# 내부 구조 검증
import zipfile
with zipfile.ZipFile(f"{zip_name}.zip") as z:
    names = z.namelist()
    root_entries = {n.split("/")[0] for n in names if n.strip("/")}
    print(f"  zip 루트 폴더: {root_entries}")
    if "model" in root_entries:
        print(f"  ✅ 올바른 구조: model/ 루트 확인")
    else:
        print(f"  ❌ 구조 오류: {root_entries} — 제출 전 수동 확인 필요")

import os
zip_mb = os.path.getsize(f"{zip_name}.zip") / 1e6
print(f"\n[INFO] 생성 완료: {zip_name}.zip ({zip_mb:.0f} MB)")